# 시작하기: Self-Managed VPC Lattice를 사용하는 Private API Gateway

이 실습에서는 [관리형 VPC 리소스 실습](../01-managed-vpc-resource/01-getting-started.ipynb)과 동일하게 모의 통합이 구성된 프라이빗 [Amazon API Gateway](https://docs.aws.amazon.com/apigateway/latest/developerguide/apigateway-private-apis.html)를 배포합니다. 다만 AgentCore가 VPC Lattice 리소스를 관리하도록 하는 대신, boto3를 사용하여 **Resource Gateway와 Resource Configuration을 직접 생성하고 관리합니다**.

### 관리형 VPC 리소스 실습과 무엇이 다른가요?

| | 관리형 | Self-Managed(이 실습) |
|---|---------|----------------------|
| **Resource Gateway** | AgentCore가 생성 | `create_resource_gateway`를 통해 직접 생성 |
| **Resource Configuration** | AgentCore가 생성 | `create_resource_configuration`을 통해 직접 생성 |
| **CreateGatewayTarget** | `managedVpcResource`(VPC, 서브넷, SG) | `selfManagedLatticeResource`(RC ARN만 지정) |
| **정리** | 대상만 삭제 | 대상, Resource Configuration, Resource Gateway 삭제 |
| **제어** | AgentCore가 수명 주기 관리 | ENI별 서브넷 배치, SG, IP 수를 직접 제어 |

Self-Managed Lattice에 대한 배경 정보는 [Self-Managed Lattice README 문서](./README.md)를 참조하세요.

## 아키텍처

![아키텍처](./images/api-gw.png)

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(VPC + AgentCore Gateway 배포)

이 실습에는 도메인 이름이나 ACM 인증서가 필요하지 않습니다.

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")
lattice = session.client("vpc-lattice")

# Cognito 클라이언트 보안 암호 가져오기
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

## 2단계: Private API Gateway 배포

이 CDK 스택은 다음을 배포합니다.
- API 키로 보호되며 모의 통합(`/health` GET, `/items` GET/POST)이 구성된 **Private API Gateway**
- 프라이빗 DNS가 활성화된 프라이빗 서브넷의 `execute-api`용 **VPC Endpoint**
- VPC CIDR에서 인바운드 HTTPS(443)를 허용하는 **보안 그룹**

VPC 엔드포인트가 프라이빗 API와 연결되면 API Gateway는 다음과 같은 특수한 DNS 이름을 생성합니다.
```
https://{api-id}-{vpce-id}.execute-api.{region}.amazonaws.com/{stage}
```

이 DNS 이름은 **퍼블릭 DNS에서 확인할 수 있으며**(VPCE 프라이빗 IP로 확인됨), 유효한 AWS 관리형 TLS 인증서를 사용합니다. 사용자 지정 도메인이나 ACM 인증서는 필요하지 않습니다.

> 이 스택은 [관리형 VPC 리소스 실습](../01-managed-vpc-resource/01-getting-started.ipynb)에서 사용하는 스택과 동일합니다. 이미 배포했다면 이 단계에서 "no changes"가 표시됩니다.

In [ ]:
!cdk deploy PrivateApigw --profile {ACCOUNT_A_PROFILE} --require-approval never --outputs-file apigw-outputs.json

In [ ]:
with open("apigw-outputs.json") as f:
    apigw_outputs = json.load(f)["PrivateApigw"]

API_ID = apigw_outputs["ApiId"]
API_KEY_ID = apigw_outputs["ApiKeyId"]
VPCE_ID = apigw_outputs["VpceId"]
VPCE_SG_ID = apigw_outputs["VpceSgId"]

API_VPCE_DNS = f"{API_ID}-{VPCE_ID}.execute-api.{REGION}.amazonaws.com"

# API 키 값 가져오기
apigw_client = session.client("apigateway")
api_key_response = apigw_client.get_api_key(apiKey=API_KEY_ID, includeValue=True)
API_KEY_VALUE = api_key_response["value"]

print(f"API ID:       {API_ID}")
print(f"VPCE ID:      {VPCE_ID}")
print(f"API-VPCE DNS: {API_VPCE_DNS}")
print(f"VPCE SG:      {VPCE_SG_ID}")

## 3단계: VPC Lattice Resource Gateway 생성

Self-Managed 모드에서는 Resource Gateway를 **직접** 생성합니다. 그러면 지정한 서브넷에 ENI가 프로비저닝되며, 이 ENI는 VPC로 들어오는 AgentCore 트래픽의 진입점 역할을 합니다.

![Resource Gateway 구성](./images/resource-gateway.png)

다음 항목을 직접 제어할 수 있습니다.
- ENI가 배치되는 서브넷
- ENI에 적용되는 보안 그룹
- ENI당 IP 수(기본값 1, 최대 62)

In [ ]:
# Resource Gateway 생성
rg_response = lattice.create_resource_gateway(
    name="self-managed-apigw-rg",
    vpcIdentifier=VPC_USW2_ID,
    subnetIds=VPC_USW2_PRIVATE_SUBNETS,
    securityGroupIds=[VPCE_SG_ID],
    ipAddressType="IPV4",
)

RESOURCE_GATEWAY_ID = rg_response["id"]
print(f"Resource Gateway ID:  {RESOURCE_GATEWAY_ID}")
print(f"Resource Gateway ARN: {rg_response['arn']}")
print(f"Status:               {rg_response['status']}")

In [ ]:
# Resource Gateway가 ACTIVE 상태가 될 때까지 대기
while True:
    rg = lattice.get_resource_gateway(resourceGatewayIdentifier=RESOURCE_GATEWAY_ID)
    status = rg["status"]
    print(f"Status: {status}")
    if status == "ACTIVE":
        print("\nResource Gateway is active!")
        break
    if status == "CREATE_FAILED":
        print(f"\nFailed: {rg}")
        break
    time.sleep(15)

## 4단계: Resource Configuration 생성

Resource Configuration은 AgentCore가 Resource Gateway를 통해 **어떤 대상에** 접근할 수 있는지 정의합니다. 전체 VPC에 대한 접근 권한을 부여하는 대신 연결 범위를 단일 엔드포인트로 제한합니다.

![Resource Configuration 구성](./images/resource-config.png)

여기서는 API-VPCE DNS 이름을 가리키는 DNS 기반 리소스 구성을 사용합니다.

In [ ]:
# Resource Configuration 생성
rc_response = lattice.create_resource_configuration(
    name="self-managed-apigw-rc",
    type="SINGLE",
    resourceGatewayIdentifier=RESOURCE_GATEWAY_ID,
    resourceConfigurationDefinition={
        "dnsResource": {
            "domainName": API_VPCE_DNS,
            "ipAddressType": "IPV4",
        }
    },
    portRanges=["443"],
)

RESOURCE_CONFIG_ARN = rc_response["arn"]
RESOURCE_CONFIG_ID = rc_response["id"]
print(f"Resource Configuration ID:  {RESOURCE_CONFIG_ID}")
print(f"Resource Configuration ARN: {RESOURCE_CONFIG_ARN}")
print(f"Status:                     {rc_response['status']}")

In [ ]:
# Resource Configuration이 ACTIVE 상태가 될 때까지 대기
while True:
    rc = lattice.get_resource_configuration(resourceConfigurationIdentifier=RESOURCE_CONFIG_ID)
    status = rc["status"]
    print(f"Status: {status}")
    if status == "ACTIVE":
        print("\nResource Configuration is active!")
        break
    if status == "CREATE_FAILED":
        print(f"\nFailed: {rc}")
        break
    time.sleep(15)

## 5단계: API Key Credential Provider 생성

관리형 VPC 리소스 실습과 마찬가지로 AgentCore가 API Gateway에 인증하려면 API Key Credential Provider가 필요합니다.

In [ ]:
cred_response = agentcore.create_api_key_credential_provider(
    name="self-managed-apigw-api-key",
    apiKey=API_KEY_VALUE,
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

## 6단계: AgentCore Gateway Target 생성

이제 `selfManagedLatticeResource`를 사용하여 Gateway Target을 생성합니다. VPC, 서브넷, SG 세부 정보는 이미 Resource Gateway에 구성했으므로 여기서는 **Resource Configuration 식별자**만 제공합니다.

![AgentCore Gateway 구성](./images/api-gw.png)

AgentCore가 Resource Configuration을 서비스 네트워크와 연결하여 엔드 투 엔드 연결을 완성합니다.

In [ ]:
# OpenAPI 스키마를 로드하고 서버 URL 주입
with open("02-self-managed-lattice/openapi-private-apigw.json") as f:
    openapi_schema = json.load(f)

TARGET_ENDPOINT = f"https://{API_VPCE_DNS}/prod"
openapi_schema["servers"] = [{"url": TARGET_ENDPOINT}]

OPENAPI_SCHEMA = json.dumps(openapi_schema)
print(f"Target endpoint: {TARGET_ENDPOINT}")

In [ ]:
response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="self-managed-apigw",
    description="Private API Gateway via self-managed VPC Lattice",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "selfManagedLatticeResource": {
            "resourceConfigurationIdentifier": RESOURCE_CONFIG_ARN,
        }
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Resource association: {target.get('privateEndpointManagedResources', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 7단계: AgentCore Gateway를 통해 API 호출

Cognito에서 액세스 토큰을 가져온 다음 Gateway를 통해 Private API Gateway의 작업을 호출합니다.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "self-managed-apigw___healthCheck", "arguments": {}},
        "id": 2,
    },
)
print("Health check:")
print(json.dumps(response.json(), indent=2))

In [ ]:
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "self-managed-apigw___listItems", "arguments": {}},
        "id": 3,
    },
)
print("Items:")
print(json.dumps(response.json(), indent=2))

## 정리

Self-Managed 모드에서는 VPC Lattice 리소스를 다음 역순으로 직접 삭제해야 합니다.
1. Gateway Target 삭제
2. Credential Provider 삭제
3. 서비스 네트워크 리소스 연결 삭제(AgentCore가 생성하며, 삭제 가능한 상태가 되기까지 몇 분 정도 걸릴 수 있음)
4. Resource Configuration 삭제
5. Resource Gateway 삭제
6. API Gateway CDK 스택 제거

> **참고:** Gateway Target을 삭제하면 AgentCore가 서비스 네트워크 리소스 연결을 비동기적으로 제거합니다. 이 작업에는 몇 분 정도 걸릴 수 있습니다. Resource Configuration을 삭제할 때 "has existing association with service networks" 오류가 발생하면 잠시 기다린 후 다시 시도하세요.

In [ ]:
# # 1단계: Gateway Target 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # 2단계: Credential Provider 삭제
# agentcore.delete_api_key_credential_provider(name="self-managed-apigw-api-key")
# print("Deleted credential provider")

# # 3단계: Resource Configuration 삭제
# # "has existing association" 오류가 발생하면 몇 분 기다린 후 다시 시도하세요.
# # AgentCore는 대상이 삭제된 후 서비스 네트워크 연결을 비동기적으로 제거합니다.
# lattice.delete_resource_configuration(resourceConfigurationIdentifier=RESOURCE_CONFIG_ID)
# print(f"Deleted Resource Configuration: {RESOURCE_CONFIG_ID}")

# # 4단계: Resource Gateway 삭제
# lattice.delete_resource_gateway(resourceGatewayIdentifier=RESOURCE_GATEWAY_ID)
# print(f"Deleted Resource Gateway: {RESOURCE_GATEWAY_ID}")

In [ ]:
# 5단계: 스택 제거(SG는 유지되며 다음 셀에서 삭제)
# !cdk destroy PrivateApigw --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # 6단계: 유지된 VPCE 보안 그룹 삭제
# # "DependencyViolation" 오류가 발생하면 ENI가 해제될 때까지 몇 분 기다리세요.
# ec2_client = session.client("ec2")
# try:
#     ec2_client.delete_security_group(GroupId=VPCE_SG_ID)
#     print(f"Deleted security group: {VPCE_SG_ID}")
# except ec2_client.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(f"SG {VPCE_SG_ID} still has dependencies. Wait a few minutes and retry.")
#     else:
#         raise